# NEW201 · Presentación final
## Problema #02 · DR Data Drifting · Grupo A

Cuaderno para **Jupyter con kernel R** (IRkernel). Reúne todo lo que se usa en la presentación: tablas, gráficos, Wilcoxon y ensambles.

**Qué se comparó** (9 configuraciones, 10 semillas cada una, AUC en validación 202107):

| variante | métodos de drift |
|---|---|
| `intrames` (sin k-means) | rank_cero_fijo, deflación IPC, deflación USD oficial, deflación USD blue |
| `kmeans_feature` (cluster y distancia como variables) | los mismos 4 métodos |
| `kmeans_drift` (rank dentro de cada cluster de cada mes) | solo rank_cero_fijo |

**Cómo está armado**
* Los resultados finales ya vienen **embebidos** en el cuaderno, así que las secciones 1 y 2 funcionan sin ningún archivo.
* Las secciones 3 y 4 (Wilcoxon contra la mejor y ensambles entre configuraciones) necesitan la carpeta `exp/NEW201`: el cuaderno la busca solo y, si no está, avisa y las saltea.
* Los gráficos y las tablas se guardan en la carpeta `figuras_presentacion/`.

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
if( !require("data.table")) install.packages("data.table")
require("data.table")

# tamaño de los gráficos dentro de Jupyter
options(repr.plot.width= 11, repr.plot.height= 6)

## 0. Dónde están los datos
El cuaderno prueba estas carpetas y se queda con la primera que exista. Si la tuya es otra, descomentar la última línea y poner la ruta.

In [ ]:
candidatas <- c(
  "/content/buckets/b1/exp/NEW201",
  "/home/ds/buckets/b1/exp/NEW201",
  "exp/NEW201",
  "NEW201",
  "../exp/NEW201"
)

dir_exp <- candidatas[ dir.exists(candidatas) ][1]

# dir_exp <- "C:/Users/TU_USUARIO/Documents/exp/NEW201"   # <- si está en otro lado

hay_datos <- !is.na(dir_exp)
if( hay_datos ) {
  message("Datos en: ", dir_exp)
} else {
  message("No encontré la carpeta exp/NEW201: se usan solo los resultados embebidos.")
}

dir_fig <- "figuras_presentacion"
dir.create(dir_fig, showWarnings= FALSE)

## Resultados finales embebidos
Son los de `resumen_configuraciones.txt` y `comparacion_intrames_vs_kmeans.txt` de la corrida final. Si esos archivos existen en la carpeta, se leen de ahí.

In [ ]:
tb_res_embebida <- fread(text= "config,dr,variante,cantidad_campos,auc_grid_mejor,num_leaves,min_data_in_leaf,feature_fraction,num_iterations,auc_val_media,auc_val_sd,auc_val_ensemble,minutos
01_rank_cero_fijo_intrames,rank_cero_fijo,intrames,272,0.9361031,256,2048,0.8,140,0.9348609,0.0006163369,0.9357579,48.83415
05_rank_cero_fijo_kmeans_feature,rank_cero_fijo,kmeans_feature,282,0.9356862,512,512,0.5,103,0.9337766,0.0011279547,0.9357594,51.67393
06_deflacion_kmeans_feature,deflacion,kmeans_feature,282,0.9345213,512,1024,0.5,96,0.9332871,0.0006720088,0.9346853,48.90823
04_deflacion_usd_blue_intrames,deflacion_usd_blue,intrames,272,0.9339983,384,2048,0.8,100,0.9330017,0.0005841661,0.9338196,47.91596
08_deflacion_usd_blue_kmeans_feature,deflacion_usd_blue,kmeans_feature,282,0.9341488,256,1024,0.5,102,0.9328350,0.0012495424,0.9341706,54.38037
07_deflacion_usd_oficial_kmeans_feature,deflacion_usd_oficial,kmeans_feature,282,0.9338847,384,1024,0.8,111,0.9325743,0.0006683231,0.9337704,50.00177
09_rank_cero_fijo_kmeans_drift,rank_cero_fijo,kmeans_drift,272,0.9337004,128,2048,0.5,190,0.9322783,0.0007172276,0.9334313,63.99852
02_deflacion_intrames,deflacion,intrames,272,0.9335813,128,512,0.8,103,0.9321522,0.0008193532,0.9332735,45.21249
03_deflacion_usd_oficial_intrames,deflacion_usd_oficial,intrames,272,0.9342053,256,2048,0.8,85,0.9320611,0.0009577375,0.9327350,45.34845
")

tb_comp_embebida <- fread(text= "dr,comparacion,auc_intrames,auc_kmeans,diferencia,semillas_que_mejoran,semillas,p_wilcoxon_pareado
rank_cero_fijo,kmeans_feature vs intrames,0.9348609,0.9337766,-0.0010842554,1,10,0.008045237
rank_cero_fijo,kmeans_drift vs intrames,0.9348609,0.9322783,-0.0025825683,0,10,0.005921537
deflacion,kmeans_feature vs intrames,0.9321522,0.9332871,0.0011349692,9,10,0.010826921
deflacion_usd_oficial,kmeans_feature vs intrames,0.9320611,0.9325743,0.0005131922,5,10,0.221271816
deflacion_usd_blue,kmeans_feature vs intrames,0.9330017,0.9328350,-0.0001666870,5,10,1.000000000
")

leer_o_embebida <- function(archivo, embebida) {
  ruta <- if( hay_datos ) file.path(dir_exp, archivo) else NA_character_
  if( !is.na(ruta) && file.exists(ruta) ) {
    message("leído de ", ruta)
    fread(ruta)
  } else {
    message("no encontré ", archivo, ": uso los resultados embebidos")
    copy(embebida)
  }
}

tb_res <- leer_o_embebida("resumen_configuraciones.txt", tb_res_embebida)
tb_comp <- leer_o_embebida("comparacion_intrames_vs_kmeans.txt", tb_comp_embebida)

setorder(tb_res, -auc_val_media)

redondear <- function(tb, d= 6) {
  x <- copy(tb)
  for( j in names(x) ) if( is.numeric(x[[j]]) ) set(x, j= j, value= round(x[[j]], d))
  x
}

## 1. Ranking de las 9 configuraciones
AUC en validación (202107): media y desvío entre las 10 semillas, y AUC del **ensamble** de las 10 semillas (que es lo que se sube a Kaggle).

In [ ]:
tb_ranking <- redondear( tb_res[, list(
  config, auc_val_media, auc_val_sd, auc_val_ensemble, auc_grid_mejor,
  num_leaves, min_data_in_leaf, feature_fraction, num_iterations, minutos
)], 6)

fwrite(tb_ranking, file.path(dir_fig, "tabla_ranking_configuraciones.csv"))
tb_ranking

In [ ]:
grafico_auc <- function() {
  g <- copy(tb_res)
  setorder(g, auc_val_media)
  n <- nrow(g)
  rango <- range(c(g$auc_val_media - g$auc_val_sd, g$auc_val_media + g$auc_val_sd, g$auc_val_ensemble))

  par(mar= c(4, 17, 3, 1))
  plot(NA, xlim= rango + c(-0.0003, 0.0003), ylim= c(-0.6, n + 0.5), yaxt= "n",
    xlab= "AUC en validación (202107)", ylab= "",
    main= "AUC por configuración: media de 10 semillas y ensamble")
  axis(2, at= 1:n, labels= g$config, las= 1, cex.axis= 0.85)
  abline(h= 1:n, col= "grey90")
  segments(g$auc_val_media - g$auc_val_sd, 1:n, g$auc_val_media + g$auc_val_sd, 1:n, lwd= 2)
  points(g$auc_val_media, 1:n, pch= 19, cex= 1.3)
  points(g$auc_val_ensemble, 1:n, pch= 17, col= "firebrick", cex= 1.3)
  # la leyenda va en la franja libre de abajo, para no tapar ninguna configuración
  legend("bottom", legend= c("media de 10 semillas ± 1 sd", "ensamble de las 10 semillas"),
    pch= c(19, 17), col= c("black", "firebrick"), bty= "n", horiz= TRUE)
}

grafico_auc()

png(file.path(dir_fig, "01_auc_por_configuracion.png"), width= 1500, height= 850, res= 130)
grafico_auc()
invisible(dev.off())

**Lectura.** Todo el rango de AUC es angosto (alrededor de 0.003). La mejor media es `01_rank_cero_fijo_intrames`; a nivel ensamble empata con `05_rank_cero_fijo_kmeans_feature`. En la 05 el ensamble sube más (+0.0020 contra +0.0009 en la 01) porque sus semillas son más variables y al promediar esa variabilidad suma diversidad.

## 2. K-means contra intra-mes
Wilcoxon **pareado por semilla** del AUC de validación. La diferencia es `k-means − intra-mes` para el mismo método de drift. El p ajustado por Holm corrige por hacer 5 comparaciones.

In [ ]:
tb_comp[, p_holm := p.adjust(p_wilcoxon_pareado, method= "holm")]
tb_comp_ver <- redondear( tb_comp[, list(
  dr, comparacion, auc_intrames, auc_kmeans, diferencia,
  semillas_que_mejoran, semillas, p_wilcoxon_pareado, p_holm
)], 6)

fwrite(tb_comp_ver, file.path(dir_fig, "tabla_kmeans_vs_intrames.csv"))
tb_comp_ver

In [ ]:
grafico_kmeans <- function() {
  x <- copy(tb_comp)
  x[, etiqueta := paste0(dr, "\n", sub(" vs intrames", "", comparacion))]
  setorder(x, diferencia)

  par(mar= c(4, 12, 3, 5), xpd= NA)
  b <- barplot(x$diferencia, horiz= TRUE, names.arg= x$etiqueta, las= 1, cex.names= 0.8,
    col= ifelse(x$diferencia > 0, "seagreen", "firebrick"),
    xlim= c(min(0, x$diferencia) * 3, max(0, x$diferencia) * 3),
    xlab= "Diferencia de AUC (k-means menos intra-mes)",
    main= "Efecto de k-means sobre el AUC de validación")
  par(xpd= FALSE)     # la línea del cero se recorta al área del gráfico
  abline(v= 0)
  par(xpd= NA)
  text(x$diferencia, b,
    labels= paste0(x$semillas_que_mejoran, "/", x$semillas, " semillas · p=", signif(x$p_wilcoxon_pareado, 2)),
    pos= ifelse(x$diferencia > 0, 4, 2), cex= 0.8)
}

grafico_kmeans()

png(file.path(dir_fig, "02_kmeans_vs_intrames.png"), width= 1500, height= 850, res= 130)
grafico_kmeans()
invisible(dev.off())

**Lectura.**
* **rank_cero_fijo:** k-means lo empeora, ya sea como variable (1 de 10 semillas mejora) o como drift por cluster (0 de 10). El rank ya normaliza la posición del cliente dentro del mes, y el cluster parece aportar poco o ruido.
* **deflación con IPC:** k-means ayuda (9 de 10 semillas). Es plausible que el cluster le devuelva información de posición relativa que la deflación sola no tiene (hipótesis, no está probada).
* **dólar oficial y dólar blue:** no hay diferencia detectable.
* Los tres resultados con p chico siguen siendo significativos con Holm.

## 3. Wilcoxon de cada configuración contra la mejor
Requiere el archivo `auc_validacion_todas_las_semillas.txt` (AUC por semilla). Se muestran dos versiones:
* **no pareado** (rank-sum), que es el principal: cada configuración usa otros campos e hiperparámetros, así que una misma semilla no es "el mismo azar";
* **pareado por semilla**, como referencia (es el que usa el cuaderno de la corrida).

Holm corrige por las 8 comparaciones.

In [ ]:
archivo_semillas <- if( hay_datos ) file.path(dir_exp, "auc_validacion_todas_las_semillas.txt") else NA_character_
hay_semillas <- !is.na(archivo_semillas) && file.exists(archivo_semillas)

if( hay_semillas ) {

  tb_semillas <- fread(archivo_semillas)    # semilla, auc_validacion, config, dr, variante

  tb_media <- tb_semillas[, list(auc_media= mean(auc_validacion),
                                 auc_sd= sd(auc_validacion),
                                 n= .N), by= config]
  setorder(tb_media, -auc_media)
  mejor <- tb_media[1, config]

  a <- tb_semillas[config == mejor][order(semilla), auc_validacion]

  tb_wil <- rbindlist(lapply(setdiff(tb_media$config, mejor), function(cfg) {
    b <- tb_semillas[config == cfg][order(semilla), auc_validacion]
    data.table(
      config= cfg,
      diferencia= mean(a) - mean(b),
      semillas_donde_gana_la_mejor= sum(a > b),
      p_no_pareado= wilcox.test(a, b, paired= FALSE, exact= FALSE)$p.value,
      p_pareado= suppressWarnings(wilcox.test(a, b, paired= TRUE, exact= FALSE)$p.value)
    )
  }))
  tb_wil[, p_no_pareado_holm := p.adjust(p_no_pareado, method= "holm")]
  tb_wil[, p_pareado_holm := p.adjust(p_pareado, method= "holm")]
  setorder(tb_wil, p_no_pareado)

  cat("Mejor configuración por AUC medio:", mejor, "\n")
  tb_wil_ver <- redondear(tb_wil, 6)
  fwrite(tb_wil_ver, file.path(dir_fig, "tabla_wilcoxon_contra_la_mejor.csv"))
  tb_wil_ver

} else {
  message("No encuentro auc_validacion_todas_las_semillas.txt: se saltea la sección 3.")
}

In [ ]:
grafico_semillas <- function() {
  g <- copy(tb_semillas)
  orden <- tb_media[order(auc_media), config]     # la mejor queda arriba
  g[, cfg := factor(config, levels= orden)]

  par(mar= c(4, 17, 3, 1))
  boxplot(auc_validacion ~ cfg, data= g, horizontal= TRUE, las= 1, cex.axis= 0.85,
    col= "grey92", outline= FALSE, ylab= "",
    xlab= "AUC en validación (202107)",
    main= "Dispersión del AUC entre las 10 semillas")
  stripchart(auc_validacion ~ cfg, data= g, vertical= FALSE, method= "jitter",
    pch= 16, cex= 0.8, col= "steelblue", add= TRUE)
}

if( hay_semillas ) {
  grafico_semillas()

  png(file.path(dir_fig, "03_auc_por_semilla.png"), width= 1500, height= 850, res= 130)
  grafico_semillas()
  invisible(dev.off())
}

**Cómo leerlo.** Un p chico dice que la diferencia de AUC es estable frente al azar del entrenamiento (la semilla), en un único mes de validación (202107). No asegura que esa configuración gane en Kaggle, que se mide por ganancia sobre 202109.

**Qué muestra el gráfico.** La 01 es muy estable: sus 10 semillas quedan en un rango angosto (≈ 0.9340 a 0.9353). La 05 es de las más dispersas: su mejor semilla (≈ 0.9357) supera a cualquiera de la 01, pero su peor (≈ 0.9318) queda muy abajo, y esa distancia entre su mejor y su peor semilla (≈ 0.004) es mayor que todo el rango de AUC medio entre configuraciones (≈ 0.003). Una semilla suelta de la 05 es una lotería; por eso conviene trabajar siempre con el ensamble de las semillas.

## 4. Ensambles entre configuraciones
Cada configuración ya es un ensamble de sus 10 semillas. Acá se promedian los **ranks** de varias configuraciones y se mide el AUC en 202107, con las probabilidades de validación guardadas (no se reentrena nada).

In [ ]:
auc_ranks <- function(prob, y) {
  r <- frank(prob, ties.method= "average")
  n1 <- sum(y == 1)
  n0 <- sum(y == 0)
  (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}

hay_validacion <- hay_semillas &&
  all( dir.exists(file.path(dir_exp, tb_media$config, "validacion")) )

if( hay_validacion ) {

  ens_config <- function(config) {
    archivos <- list.files(file.path(dir_exp, config, "validacion"), full.names= TRUE)
    x <- rbindlist(lapply(archivos, fread))
    x <- x[, list(prob= mean(prob)), by= list(numero_de_cliente, clase_ternaria)]
    setorder(x, numero_de_cliente)
    x[, clase01 := as.integer(clase_ternaria %in% c("BAJA+1", "BAJA+2"))]
    x[, rk := frank(prob) / .N]
    x[, list(numero_de_cliente, clase01, rk)]
  }

  lista <- lapply(tb_media$config, ens_config)
  names(lista) <- tb_media$config

  for( x in lista ) stopifnot( identical(x$numero_de_cliente, lista[[1]]$numero_de_cliente) )
  y <- lista[[1]]$clase01

  auc_combo <- function(cfgs) {
    auc_ranks(rowMeans(sapply(lista[cfgs], function(x) x$rk)), y)
  }

  nombres <- tb_media$config
  elegidas <- c("01_rank_cero_fijo_intrames",
                "05_rank_cero_fijo_kmeans_feature",
                "06_deflacion_kmeans_feature")

  tb_combo <- data.table(
    combinacion= c("la mejor sola", "top 2", "top 3", "top 5", "las 9", "01 + 05 + 06"),
    auc_ensemble= c(
      auc_combo(nombres[1]),
      auc_combo(nombres[1:2]),
      auc_combo(nombres[1:3]),
      auc_combo(nombres[1:5]),
      auc_combo(nombres),
      auc_combo(elegidas)
    )
  )

  tb_combo_ver <- redondear(tb_combo, 6)
  fwrite(tb_combo_ver, file.path(dir_fig, "tabla_ensambles_entre_configuraciones.csv"))
  tb_combo_ver

} else {
  message("No encuentro las carpetas 'validacion' de cada configuración: se saltea la sección 4.")
}

## 5. Conclusiones para la presentación

1. **La mejor base es `01_rank_cero_fijo_intrames`**: mayor AUC medio (0.9349), dispersión entre semillas muy baja (la segunda menor, apenas por encima de la 04) y es la más simple (272 campos, sin k-means).
2. **El rank de cero fijo no necesita k-means.** Con k-means el AUC por semilla baja; a nivel ensamble queda empatado con la 01.
3. **En deflación con IPC, k-means ayuda** (9 de 10 semillas), pero no alcanza a los métodos de rank.
4. **Dólar oficial y dólar blue**: sin diferencias detectables entre intra-mes y k-means.
5. **La 01 le gana a las otras 8 configuraciones** (Wilcoxon no pareado con Holm: p ajustado entre 0.0015 y 0.021; gana en 9 o 10 de las 10 semillas). La más cercana es la 05 (p = 0.021).
6. **El mejor ensamble entre configuraciones es 01 + 05** (AUC 0.93624 contra 0.93576 de la 01 sola). Sumar más configuraciones lo empeora (top 3: 0.93606, top 5: 0.93565, las 9: 0.93540). Esa combinación se eligió mirando el mismo mes de validación, así que la mejora es optimista.
7. **Todas las diferencias son chicas** (rango de AUC ≈ 0.003) y salen de un único mes de validación.
8. **Limitaciones.**
   * NEW201 no incluye el baseline `ninguno`, así que no se puede afirmar que corregir el drift supere a no corregirlo.
   * El AUC no es la ganancia: el orden final se define con la ganancia en Kaggle.

**Próximos pasos:** ensamblar entre configuraciones (sección 4), probar más historia en las variables (lag3, lag6, medias móviles, pendiente), ampliar el grid (varias configuraciones eligieron `min_data_in_leaf` = 2048, el borde) y elegir el corte de envíos por ganancia.

In [ ]:
list.files(dir_fig)